# 03 — From elasticity to optimal price

Given a corrected elasticity ε (< −1) and a unit cost `c`, constant-elasticity margin
`(p−c)·q(p)` is maximized at the closed form **`p* = c·ε/(ε+1)`**. We:

1. confirm closed-form == numeric optimum,
2. score the **counterfactual margin uplift** of `p*` vs the observed price,
3. run a few cost scenarios (the data has no true unit cost), and
4. show how the recommendation moves as ε swings from the naive to the corrected estimate.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
from src.data import load_prepared
from src.elasticity import fixed_effects
from src.optimize import optimal_price, margin_uplift, fit_intercept, optimize_numeric

df = load_prepared()
eps = fixed_effects(df).epsilon
print(f"corrected elasticity eps = {eps:.3f}")

In [ ]:
# closed form vs numeric, anchored to a representative (price, qty) point
p_obs, q_obs = df["unit_price"].median(), df["qty"].median()
A = fit_intercept(p_obs, q_obs, eps)
c = 0.6 * p_obs  # ASSUMED unit cost = 60% of median price (scenario; data has no cost)
print(f"closed-form p* = {optimal_price(c, eps):.2f}   numeric p* = {optimize_numeric(c, eps, A):.2f}")

In [ ]:
# cost-scenario sweep: optimal price and counterfactual margin uplift vs observed
rows = []
for cost_frac in [0.4, 0.5, 0.6, 0.7]:
    c = cost_frac * p_obs
    res = margin_uplift(c, eps, p_obs, q_obs)
    rows.append({"cost_frac": cost_frac, "cost": round(c, 2),
                 "p_obs": round(res.price_obs, 2), "p_star": round(res.price_star, 2),
                 "uplift_%": round(res.uplift_pct, 1)})
pd.DataFrame(rows)

## The naive-vs-corrected punchline
Same cost, two elasticities: the naive ε (≈ −0.13) isn't even elastic, so `p* = c·ε/(ε+1)` has **no
interior optimum** — it would tell you to raise price without limit. Only the corrected ε yields a
sane recommendation. That is the margin cost of pricing on the wrong number, stated concretely.

**Caveats:** single-product, static optimization; constant-elasticity form; assumed cost;
ignores competitor reaction and cross-product cannibalization (v2).